# 🚨 Обработка исключений в ООП

<div style="background: linear-gradient(135deg, #ff6b6b 0%, #ee5a24 100%); padding: 20px; border-radius: 10px; color: white;">
  <h2 style="color: white;">Как сделать ваш код надёжнее с помощью исключений</h2>
  <p>Исключения — это не просто ошибки. В ООП они становятся частью интерфейса класса, позволяя优雅но сообщать о проблемах и разделять логику обработки ошибок.</p>
  <p>Сегодня вы научитесь:</p>
  <ul>
    <li>Создавать собственные иерархии исключений</li>
    <li>Использовать <code>try/except/else/finally</code> грамотно</li>
    <li>Применять лучшие практики обработки ошибок в классах</li>
  </ul>
</div>

## 🎯 Цели лекции

- Понять механизм исключений в Python и его роль в ООП
- Научиться создавать иерархии пользовательских исключений
- Освоить конструкцию `try/except/else/finally` во всех деталях
- Узнать лучшие практики: когда перехватывать, а когда позволить исключению всплыть
- Реализовать классы, которые сообщают об ошибках через исключения, а не через коды возврата

## 📚 План лекции

1. **Почему исключения лучше кодов ошибок?** — философия ООП
2. **Базовый синтаксис** — `try`, `except`, `else`, `finally`
3. **Иерархия встроенных исключений** — `BaseException`, `Exception`, `ArithmeticError` и т.д.
4. **Пользовательские исключения** — как создать и когда это нужно
5. **Исключения в методах классов** — проектирование надёжных API
6. **Лучшие практики** — конкретность, не глотать, логировать, использовать `finally`
7. **Примеры из реальной ООП** — банковский счёт, валидатор, работа с файлами
8. **Итоги и контрольные вопросы**

In [1]:
# Вспомним, как это было без исключений (коды ошибок)
class OldCalculator:
    def divide(self, a, b):
        if b == 0:
            return None  # или -1, или другой код ошибки
        return a / b

calc = OldCalculator()
result = calc.divide(10, 0)
if result is None:
    print("Ошибка: деление на ноль")
else:
    print(result)
# Проблема: код ошибки можно проигнорировать, и программа продолжит работу с некорректными данными

Ошибка: деление на ноль


## 1. Почему исключения лучше кодов ошибок?

В процедуральном стиле часто возвращают специальные значения (например, `-1`, `None`) для обозначения ошибки. Недостатки:
- Можно забыть проверить код ошибки
- Код ошибки не содержит информации о причине
- Затрудняется передача ошибки через несколько уровней вызовов

**Исключения** в ООП:
- Разделяют нормальный поток выполнения и обработку ошибок
- Нельзя проигнорировать — если не перехватить, программа остановится
- Могут содержать подробную информацию (текст, атрибуты)
- Поддерживают иерархию (можно перехватывать общие или конкретные типы)

В классах исключения становятся частью контракта: метод говорит "я могу выбросить исключение такого-то типа".

## 2. Базовый синтаксис try/except/else/finally

```python
try:
    # код, который может вызвать исключение
    risky_operation()
except SomeException as e:
    # обработка конкретного исключения
    print(f"Ошибка: {e}")
except AnotherException:
    # обработка другого типа
    handle_another()
except:   # плохая практика — перехватывает всё
    print("Какая-то ошибка")
else:
    # выполняется, если исключения не было
    print("Всё хорошо")
finally:
    # выполняется всегда (очистка ресурсов)
    cleanup()

In [2]:
# Демонстрация всех блоков
def divide(a, b):
    try:
        result = a / b
    except ZeroDivisionError as e:
        print(f"Поймали ZeroDivisionError: {e}")
        return None
    else:
        print("Деление успешно, else выполнен")
        return result
    finally:
        print("finally выполняется всегда")

print(divide(10, 2))
print(divide(10, 0))

Деление успешно, else выполнен
finally выполняется всегда
5.0
Поймали ZeroDivisionError: division by zero
finally выполняется всегда
None


## 3. Иерархия встроенных исключений

Все исключения наследуются от `BaseException`. Но перехватывать `BaseException` не рекомендуется — это прервет даже `KeyboardInterrupt` и `SystemExit`.

Обычно перехватывают `Exception` — от него наследуются все «обычные» ошибки.


In [3]:
# Пример работы с иерархией
try:
    x = int("abc")
except ValueError as e:
    print(f"ValueError: {e}")
except Exception as e:
    print(f"Другое исключение: {e}")

# Можно перехватывать родительский класс — поймает и дочерние
try:
    result = 10 / 0
except ArithmeticError as e:
    print(f"Арифметическая ошибка: {e}")

ValueError: invalid literal for int() with base 10: 'abc'
Арифметическая ошибка: division by zero


## 4. Пользовательские исключения

Создавайте свои классы исключений, наследуя от `Exception` (или от одного из его подклассов).

### Зачем?
- Сделать API класса более выразительным
- Передавать дополнительную информацию (например, значение, вызвавшее ошибку)
- Группировать ошибки по смыслу

### Соглашения:
- Имя класса заканчивается на `Error`
- Обычно достаточно пустого тела (документация)
- Можно добавить атрибуты и метод `__init__`

In [7]:
# Создание простого пользовательского исключения
class InsufficientFundsError(Exception):
    """Выбрасывается, когда на счёте недостаточно средств."""
    pass

class NegativeAmountError(ValueError):
    """Выбрасывается при попытке внести отрицательную сумму."""
    def __init__(self, amount):
        self.amount = amount
        super().__init__(f"Сумма не может быть отрицательной: {amount}")

# Использование в классе
class BankAccount:
    def __init__(self, balance=0):
        self._balance = balance

    def withdraw(self, amount):
        if amount < 0:
            raise NegativeAmountError(amount)
        if amount > self._balance:
            raise InsufficientFundsError(f"Недостаточно средств. Доступно: {self._balance}")
        self._balance -= amount
        return self._balance

acc = BankAccount(-100)
try:
    acc.withdraw(-2000)
except InsufficientFundsError as e:
    print(f"Ошибка: {e}")

NegativeAmountError: Сумма не может быть отрицательной: -2000

## 5. Исключения в методах классов: проектирование контракта

Хорошо спроектированный класс явно документирует, какие исключения могут быть выброшены.

### Рекомендации:
- Не перехватывайте исключения внутри метода, если не можете их обработать осмысленно
- Если метод не может выполнить свою задачу, пусть выбросит исключение (не возвращает `None`)
- Пользовательские исключения делают API самодокументируемым

### Пример: класс валидации

In [10]:
class ValidationError(Exception):
    """Базовое исключение для ошибок валидации."""
    pass

class TooShortError(ValidationError):
    def __init__(self, field, min_length):
        self.field = field
        self.min_length = min_length
        super().__init__(f"{field} должен быть длиннее {min_length} символов")

class TooLongError(ValidationError):
    def __init__(self, field, max_length):
        self.field = field
        self.max_length = max_length
        super().__init__(f"{field} не должен превышать {max_length} символов")

class User:
    def __init__(self, username):
        self.username = username
        self.validate()

    def validate(self):
        if len(self.username) < 3:
            raise TooShortError("username", 3)
        if len(self.username) > 6:
            raise TooLongError("username", 6)

# Использование
try:
    u = User("axfgdfgfghfghjfgb")
except ValidationError as e:
    print(f"Ошибка создания пользователя: {e}")

Ошибка создания пользователя: username не должен превышать 6 символов


## 6. Лучшие практики обработки исключений

### ✅ DO (делайте)
1. **Перехватывайте конкретные исключения** — избегайте голого `except:`
2. **Используйте `else`** — для кода, который должен выполниться только при отсутствии ошибки
3. **Используйте `finally`** — для освобождения ресурсов (закрытие файлов, соединений)
4. **Создавайте свои иерархии исключений** — чтобы можно было перехватывать общий тип
5. **Документируйте исключения в docstring** (что и когда выбрасывает метод)
6. **Логируйте исключения** перед тем, как перехватить и проглотить

### ❌ DON'T (не делайте)
1. **Не перехватывайте `BaseException`** (если только не знаете, зачем)
2. **Не глотайте исключения молча** — `except: pass` — зло
3. **Не бросайте исключения от базовых типов** (`raise Exception("message")`) — лучше пользовательское или конкретное встроенное
4. **Не используйте исключения для управления потоком** (как `goto`) — это медленно и запутывает
5. **Не возвращайте коды ошибок вместе с исключениями** — выберите что-то одно

In [12]:
# Пример плохой практики (глотание исключений)
# try:
#     risky_operation()
# except:
#     pass  # ошибка скрыта, трудно отлаживать

# Хорошо: логировать и/или пробрасывать дальше
import logging
logging.basicConfig(level=logging.ERROR)
try:
    risky_operation()
except ValueError as e:
    logging.error(f"Ошибка: {e}")
    raise  # пробрасываем дальше

NameError: name 'risky_operation' is not defined

## 7. Практические примеры из ООП

### Пример 1. Банковский счёт с пользовательскими исключениями

In [ ]:
class BankError(Exception):
    """Базовое исключение для банковских операций."""
    pass

class NegativeAmountError(BankError):
    pass

class InsufficientFundsError(BankError):
    def __init__(self, balance, requested):
        self.balance = balance
        self.requested = requested
        super().__init__(f"Недостаточно средств: запрошено {requested}, доступно {balance}")

class AccountLockedError(BankError):
    pass

class BankAccount:
    def __init__(self, owner, initial_balance=0):
        self.owner = owner
        self._balance = initial_balance
        self._locked = False

    def deposit(self, amount):
        if self._locked:
            raise AccountLockedError("Счёт заблокирован")
        if amount <= 0:
            raise NegativeAmountError(f"Сумма депозита должна быть положительной: {amount}")
        self._balance += amount
        return self._balance

    def withdraw(self, amount):
        if self._locked:
            raise AccountLockedError("Счёт заблокирован")
        if amount <= 0:
            raise NegativeAmountError(f"Сумма снятия должна быть положительной: {amount}")
        if amount > self._balance:
            raise InsufficientFundsError(self._balance, amount)
        self._balance -= amount
        return self._balance

    @property
    def balance(self):
        return self._balance

# Демонстрация
acc = BankAccount("Alice", 500)
try:
    acc.withdraw(600)
except InsufficientFundsError as e:
    print(f"Ошибка: {e}")
    print(f"Баланс: {e.balance}, запрошено: {e.requested}")

### Пример 2. Контекстный менеджер с обработкой исключений

In [ ]:
class FileProcessor:
    def __init__(self, filename, mode='r'):
        self.filename = filename
        self.mode = mode
        self.file = None

    def __enter__(self):
        try:
            self.file = open(self.filename, self.mode)
        except FileNotFoundError as e:
            raise FileNotFoundError(f"Не удалось открыть {self.filename}: {e}")
        except PermissionError as e:
            raise PermissionError(f"Нет прав на чтение {self.filename}: {e}")
        return self.file

    def __exit__(self, exc_type, exc_val, exc_tb):
        if self.file:
            self.file.close()
        if exc_type is not None:
            print(f"При работе с файлом возникла ошибка: {exc_val}")
        return False  # не подавляем исключение

# Использование
try:
    with FileProcessor("nonexistent.txt") as f:
        data = f.read()
except FileNotFoundError as e:
    print(f"Поймали исключение: {e}")

### Пример 3. Валидация данных с цепочкой исключений (raise from)

In [ ]:
class DataValidationError(Exception):
    pass

def load_user_data(data):
    try:
        name = data['name']
        age = int(data['age'])
    except KeyError as e:
        raise DataValidationError(f"Отсутствует поле {e}") from e
    except ValueError as e:
        raise DataValidationError(f"Некорректный возраст: {e}") from e
    return name, age

try:
    user = load_user_data({'name': 'Bob', 'age': 'не число'})
except DataValidationError as e:
    print(f"Ошибка: {e}")
    print(f"Причина: {e.__cause__}")  # исходное ValueError

## 8. Итоги и ключевые моменты

- **Исключения** — механизм, встроенный в ООП Python. Они образуют иерархию.
- **Пользовательские исключения** создаются наследованием от `Exception` и делают API классов чище.
- **Конструкция `try/except/else/finally`** позволяет гибко управлять ошибками:
  - `except` — перехват и обработка
  - `else` — код, если ошибок не было
  - `finally` — всегда выполняется (закрытие ресурсов)
- **Лучшие практики**:
  - Перехватывайте конкретные типы исключений
  - Не глотайте исключения молча
  - Используйте `raise from` для сохранения цепочки ошибок
  - Документируйте исключения в классах
